In [1]:
import cv2
import numpy as np

def deskew_plate(plate):

    gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)

    blur = cv2.GaussianBlur(gray, (3,3), 0)

    edges = cv2.Canny(blur, 50, 150)

    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 180,
        threshold=50,
        minLineLength=50,
        maxLineGap=10
    )

    if lines is None:
        return plate

    angles = []

    for line in lines:

        x1, y1, x2, y2 = line[0]

        angle = np.degrees(
            np.arctan2(y2 - y1, x2 - x1)
        )

        # chỉ lấy line gần ngang
        if -30 < angle < 30:
            angles.append(angle)

    if len(angles) == 0:
        return plate

    median_angle = np.median(angles)

    # nếu góc quá nhỏ thì bỏ
    if abs(median_angle) < 2:
        return plate

    (h, w) = plate.shape[:2]

    center = (w // 2, h // 2)

    M = cv2.getRotationMatrix2D(
        center,
        median_angle,
        1.0
    )

    rotated = cv2.warpAffine(
        plate,
        M,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE
    )

    return rotated

In [2]:
def sort_ocr_text(result_ocr):
    combined = []

    for res in result_ocr:

        # lấy text + polygon
        texts = res.get('rec_texts', [])
        polys = res.get('rec_polys', [])

        for text, poly in zip(texts, polys):

            # lấy tọa độ góc trái trên
            x = poly[0][0]
            y = poly[0][1]

            combined.append((y, x, text))

    # ===== sort =====
    # ưu tiên theo y trước (dòng)
    # rồi theo x (trái -> phải)
    combined.sort(key=lambda item: (item[0] // 20, item[1]))

    final_text = " ".join([item[2] for item in combined])

    return final_text

In [3]:
def get_best_text(history):

    if len(history) == 0:
        return ""

    counter = Counter(history)

    best_text = counter.most_common(1)[0][0]

    return best_text

In [4]:
def center(box):

    x1, y1, x2, y2 = box

    return ((x1+x2)//2, (y1+y2)//2)

In [5]:
import math

def distance(c1, c2):

    return math.sqrt(
        (c1[0]-c2[0])**2 +
        (c1[1]-c2[1])**2
    )

In [6]:
from ultralytics import YOLO
from paddleocr import PaddleOCR
from collections import defaultdict,Counter
import cv2

model = YOLO("../model/best_plate_detection.pt")
ocr = PaddleOCR(use_angle_cls=True, lang='en')

cap = cv2.VideoCapture("../samples/video.mp4")
frame_id =0

ocr_history = defaultdict(list)
tracks = {}
next_track_id = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_id += 1
    if frame_id % 5 != 0:
        continue
    results = model(frame)

    for r in results:
        boxes = r.boxes.xyxy

        for box in boxes:
            x1, y1, x2, y2 = map(int, box)

            plate = frame[y1:y2, x1:x2]
            plate = cv2.resize(plate, None, fx=2, fy=2)

            if plate.shape[0] != 0 and plate.shape[1] != 0:
                plate = deskew_plate(plate)

            result_ocr = ocr.predict(plate)

            text = ""
            if result_ocr:
                text = sort_ocr_text(result_ocr)
            current_center = center((x1, y1, x2, y2))

            matched_id = None
            global next_track_id

            for track_id, old_center in tracks.items():

                if distance(current_center, old_center) < 100:

                    matched_id = track_id
                    break
            if matched_id is None:

                matched_id = next_track_id

                next_track_id += 1

            tracks[matched_id] = current_center

            if text.strip() != "":

                ocr_history[matched_id].append(
                    text.strip()
                )
                ocr_history[matched_id] = (
                    ocr_history[matched_id][-20:]
                )

            best_text = get_best_text(
                ocr_history[matched_id]
            )
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)

            cv2.putText(frame, best_text, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.9, (0,255,0), 2)

    cv2.imshow("Video", frame)

    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()

/Users/vi/miniconda3/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/vi/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/5l/xvvkbgln1qz502f2b5fvbfj40000gn/T/ipykernel_2345/3244132774.py:7: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=True, lang='en')
/Users/vi/miniconda3/lib/python3.13/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https:


0: 544x960 2 plates, 179.9ms
Speed: 6.6ms preprocess, 179.9ms inference, 6.3ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 197.6ms
Speed: 2.6ms preprocess, 197.6ms inference, 1.5ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 157.6ms
Speed: 3.0ms preprocess, 157.6ms inference, 1.0ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 144.3ms
Speed: 2.4ms preprocess, 144.3ms inference, 0.6ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 147.3ms
Speed: 2.6ms preprocess, 147.3ms inference, 0.6ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 144.7ms
Speed: 2.9ms preprocess, 144.7ms inference, 0.4ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 142.3ms
Speed: 2.4ms preprocess, 142.3ms inference, 0.6ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 3 plates, 130.5ms
Speed: 2.6ms preprocess, 130.5ms inference, 0.4ms postprocess per image at